# Census Data Imports (Work in Progress)

By Kenneth Burchfiel

Released under the MIT License

The US Census is a fantastic source of free demographic data. Thankfully, we can easily access large amounts of this data at once via Python, as demonstrated by this notebook. The following code will retrieve data that can help answer two specific questions: (1) What would be the best place for college grads to raise a family? and (2) What is the correlation between marriage and poverty rates?

## Question 1: Deciding where to move to start a family

Let's say that some NVCU graduating seniors are interested in settling down and raising a family a few years after they graduate. Because they'd prefer to live in a growing region rather than a declining one, they want to know which areas have seen the highest growth rates in recent years. They'd like to see this data both for all residents within each county *and* those aged 25-29.

In order to answer these questions, we'll use the Census API to retrieve US county population growth data for a selected set of years. We'll then use this data to calculate population growth rates across multiple periods.

## Question 2: Understanding the correlation between marriage and poverty rates

NVCU's College of Social Work is committed to finding ways to reduce the poverty rate in the United States. As part of this mission, they would like to determine whether married couples are less likely to live below the poverty line than are non-married couples.\* They are especially interested in child poverty, so they'd also like to know whether married couples with kids have a lower poverty rate than do non-married households with kids.

*(Although Census data can help us determine the relative poverty rates between married and non-married households, it can't answer the more important question: does getting married actually *reduce* poverty? After all, it's possible that the causal path between marriage and poverty may run in reverse: individuals who are better off financially might be more likely to marry. In other words, poverty might reduce the marriage rate, not the other way around. The College of Social Work is aware of the danger of conflating correlation and causation, but they believe that this demographic data will still be worth exploring.)*

We'll also create choropleth maps in the Mapping section of Python for Nonprofits that visualize these Census data.


## An introduction to the American Community Survey

Many Americans probably associate the US Census Bureau with its decennial Census. However, the Census Bureau also conducts the [American Community Survey](https://www.census.gov/data/developers/data-sets/acs-5year.html) each year, making it a great resource for recent demographic data.

This notebook will source data from the American Community Survey's 5-year estimates, which show an average of results for the past 5 years. (For example, the 2022 ACS5 dataset shows results between 2018 and 2022). The [1-year ACS estimates](https://www.census.gov/data/developers/data-sets/acs-1year.html) offer results for a more recent timeframe; however, because the 5-year estimates are sourced from a larger pool of data, they may be more reliable (especially for smaller regions). In addition, 1-year estimates aren't available for certain regions, such as zip codes.

For the sake of brevity, I'll often refer to the American Community Survey's 5-year estimates as the 'ACS5' survey.

# Part 1: Introducing the Census API and Answering Our First Question

In the process of working to answer the first question (regarding where to move to start a family), we'll also explore the Census API's capabilities and craft a Python function that will use it to efficiently retrieve data.

Importing relevant libraries and setting two configuration variables:

In [1]:
import time
program_start_time = time.time()
import pandas as pd
import numpy as np
from iteration_utilities import duplicates
pd.set_option('display.max_columns', 1000)
import lxml # Necessary for reading online HTML tables into Pandas

render_for_pdf = False
if render_for_pdf == True:
    pd.set_option('display.max_columns', 4)


latest_acs5_year = 2022 # By updating this variable when future American 
# Community Surveys get released, you should be able to retrieve the most
# recent data possible. (If changes to the survey's format are made,
# however, updates to the scripts may be necessary.)
download_variable_list = False # If set to True, a new list of variables
# will be downloaded from the Census API website. If False, this list of 
# variables will instead be read in from a local .csv copy (thus saving
# processing time).

## Importing a Census API Key

You can obtain a free Census API key [at this website](https://api.census.gov/data/key_signup.html). The following cell imports my own personal key, so you'll need to replace this code with one that loads in your own API key.

In [2]:
with open ('census_api_key_path.txt') as file:
    key_path = file.read()
with open(key_path) as file:
    key = file.read()

The Census offers detailed API documentation that makes retrieving data from it relatively straightforward. For instance, you'll probably find the [Census Data API User Guide](https://www.census.gov/content/dam/Census/data/developers/api-user-guide/api-guide.pdf) to be helpful in applying the Census API.

[This list](https://api.census.gov/data/2022/acs/acs5/examples.html) of ACS5 API call examples is another great resource. One of the sample URLs shown on this page for retrieving county-level data appears as follows:

https://api.census.gov/data/2022/acs/acs5?get=NAME,B01001_001E&for=county:*&key=YOUR_KEY_GOES_HERE

'B01001_001E' refers to the total population estimates for a given county. We can find this out by going to the [2022 ACS5's Detailed Tables page for the](https://api.census.gov/data/2022/acs/acs5/variables.html) and navigating to the row with a 'Name' value of 'B01001_001E'. This link, which may take a little while to load, is available on the [2022 ACS5 API Documentation Page](https://www.census.gov/data/developers/data-sets/acs-5year.html).

If you replace the 'YOUR_KEY_GOES_HERE' component of the URL with your actual key, then enter this link into your web browser, you'll receive a very long list of counties, population values, and state and county codes. The top of the list for the 2022 ACS5 looked like this:

```
[["NAME","B01001_001E","state","county"],
["Autauga County, Alabama","58761","01","001"],
["Baldwin County, Alabama","233420","01","003"],
["Barbour County, Alabama","24877","01","005"],
["Bibb County, Alabama","22251","01","007"],
["Blount County, Alabama","59077","01","009"],
["Bullock County, Alabama","10328","01","011"],
```

We can use `pd.read_json()` to easily read this same data into a DataFrame:

In [3]:
df_results = pd.read_json(
    f'https://api.census.gov/data/{latest_acs5_year}/\
acs/acs5?get=NAME,B01001_001E&for=county:*&key={key}')
# read_json documentation:
# https://pandas.pydata.org/pandas-docs/stable/reference/api/
# pandas.read_json.html
df_results.head()

,0,1,2,3
0,NAME,B01001_001E,state,county
1,"Autauga County, Alabama",58761,01,001
2,"Baldwin County, Alabama",233420,01,003
3,"Barbour County, Alabama",24877,01,005
4,"Bibb County, Alabama",22251,01,007


At this point, the DataFrame's columns are [0, 1, 2, 3], whereas the columns we want to use are stored within the first row. The following code sets these row values as our column values, then deletes this row:

In [4]:
df_results.columns = df_results.iloc[0]
df_results.drop(0, inplace = True)

df_results.head()

,NAME,B01001_001E,state,county
1,"Autauga County, Alabama",58761,01,001
2,"Baldwin County, Alabama",233420,01,003
3,"Barbour County, Alabama",24877,01,005
4,"Bibb County, Alabama",22251,01,007
5,"Blount County, Alabama",59077,01,009


## Retrieving our Data

In order to determine which variable codes to enter into our script, we'll first need to review a list of all American Community Survey variables and the overall groups into which they fit. This list is available on the Census website ([here's the copy for 2022](https://api.census.gov/data/2022/acs/acs5/variables.html)), but we can also use Pandas to import them into a DataFrame, as shown below.

In [5]:
if download_variable_list == True:
    df_variables_page = pd.read_html(
        f'https://api.census.gov/data/{latest_acs5_year}/acs/acs5/variables.html')[0] 
    # [0] selects the first HTML table found on this page.
    # See https://pandas.pydata.org/pandas-docs/stable/reference/api/
    # pandas.read_html.html
    # for more information on pd.read_html().
        
    # Some rows in this table contain items other than demographic 
    # variables (e.g. region names). We can exclude them by selecting 
    # only rows that begin with 'Estimate'. (Another option would have 
    # been to filter out rows with N/A 'Group' entries (i.e. 
    # df_variables.query("Group.isna() == False")), 
    # but this would have left a couple non-variable rows in place.
    
    df_variables = df_variables_page[
    df_variables_page['Label'].str[0:8] == 'Estimate'].copy(
    ).reset_index(drop=True)
    # Saving this table to a local .csv file:
    df_variables.to_csv(f'Datasets/{latest_acs5_year}_variables.csv', 
    index = False)
else: # Reading a local copy of this dataset instead, which should 
    # take much less time. 
    df_variables = pd.read_csv(
        f'Datasets/{latest_acs5_year}_variables.csv')
df_variables.head()

,Name,Label,Concept,Required,Attributes,Limit,Predicate Type,Group,Unnamed: 8
0,B01001_001E,Estimate!!Total:,Sex by Age,not required,"B01001_001EA, B01001_001M, B01001_001MA",0,int,B01001,NaN
1,B01001_002E,Estimate!!Total:!!Male:,Sex by Age,not required,"B01001_002EA, B01001_002M, B01001_002MA",0,int,B01001,NaN
2,B01001_003E,Estimate!!Total:!!Male:!!Under 5 years,Sex by Age,not required,"B01001_003EA, B01001_003M, B01001_003MA",0,int,B01001,NaN
3,B01001_004E,Estimate!!Total:!!Male:!!5 to 9 years,Sex by Age,not required,"B01001_004EA, B01001_004M, B01001_004MA",0,int,B01001,NaN
4,B01001_005E,Estimate!!Total:!!Male:!!10 to 14 years,Sex by Age,not required,"B01001_005EA, B01001_005M, B01001_005MA",0,int,B01001,NaN


With over 28,000 individual variables, it could take a very long time to identify the items you'd like to retrieve from the Census. We can make this search process somewhat easier by creating a separate *groups* table that shows only unique group names and their written descriptions (e.g. 'Sex by Age').

In [6]:
df_groups = df_variables.drop_duplicates(
    'Group')[['Concept', 'Group']].copy(
    ).reset_index(drop=True)
df_groups.head()

,Concept,Group
0,Sex by Age,B01001
1,Sex by Age (White Alone),B01001A
2,Sex by Age (Black or African American Alone),B01001B
3,Sex by Age (American Indian and Alaska Native ...,B01001C
4,Sex by Age (Asian Alone),B01001D


We'll save this group table to a local .csv file as well:

In [7]:
df_groups.to_csv(f'Datasets/{latest_acs5_year}_groups.csv', 
                 index = False)

In order to find variables of interest, I recommend first searching for keywords of interest within the group table (which is much smaller in size) in order to identify relevant group IDs. Next, you can search for those group IDs inside the variables table in order to find the exact metrics to request from the Census API.

The following table stores variables for three separate groups: (1) the total population; (2) all males aged 25 to 29 years; and (3) all females aged 25 to 29 years. (The B01001 table that stores these values didn't have an entry for all people aged 25 to 29; we'll get around this limitation by retrieving sex-specific population totals within this age group, then adding those totals together.)

In [8]:
grad_destinations_variable_list = [
    'B01001_001E', 'B01001_011E',
    'B01001_035E']

The demographic columns in the Census API's output are labeled with their variable names (e.g. 'B01001_001E'). These names are concise, but you'll need a copy of the original variable list to interpret them. Therefore, I chose to replace these column names with a combination of the 'Label', 'Concept', and 'Name' entries in the original variable list. These column names are very long, but they do make the output easier to interpret (while also preserving the original names for reference). 

In addition, if the description corresponding to a variable name happens to change from one year to another, the use of aliases will help you identify that change. (This will help prevent you from treating two different data types that happened to use the same variable code in different years as equal. However, I would imagine that the Census wouldn't repurpose variable codes in this way.)

The following function assists with this replacement by creating a dictionary whose keys are the original field names (e.g. 'B0101_001E') and whose values are the replacement names (e.g. 'Sex by Age_Estimate!!Total:_B01001_001E').

In [9]:
def create_variable_aliases(df_variables, variable_list):
    '''This function creates a dictionary whose keys are 
    the original 'Name' values (e.g. 'B001_001E') within a variable
    list on the Census API website and whose values are the replacement 
    names (e.g. 'Sex by Age_Estimate!!Total:_B01001_001E').
    This resulting dictionary can then be passed to a df.rename() call
    within retrieve_census_data() in order to make the output of that
    function easier to interpret.
    
    df_variables: A DataFrame containing a list of Census variables. For
    an example of this list for the 2022 American Community Survey (5-Year 
    Estimates), visit: 
    https://api.census.gov/data/2022/acs/acs5/examples.html .
    
    variable_list: The list of variables to rename 
    (e.g. ['B01001_001E', 'B01001_002E']).
    '''
    # Creating a DataFrame that contains the information needed for the
    # updated column names:
    df_aliases = df_variables.query(
        "Name in @variable_list")[['Name', 'Label', 'Concept']].copy()
    # Creating a new 'Description' column that will replace the original
    # output field names:
    df_aliases['Description'] = (df_aliases['Concept'] 
                                 + '_' + df_aliases['Label'] 
                                 + ' (' + df_aliases['Name'] + ')')
    # Creating a dictionary whose keys are the original field names and 
    # whose values are the new 'Description' entries that were 
    # just created:
    alias_dict = df_aliases.set_index('Name').to_dict()['Description']
    # See https://pandas.pydata.org/pandas-docs/stable/reference/api/
    # pandas.DataFrame.to_dict.html
    return alias_dict

Creating our aliases:

In [10]:
grad_destinations_alias_dict = create_variable_aliases(
    df_variables = df_variables, 
    variable_list = grad_destinations_variable_list)
grad_destinations_alias_dict

{'B01001_001E': 'Sex by Age_Estimate!!Total: (B01001_001E)',
 'B01001_011E': 'Sex by Age_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)',
 'B01001_035E': 'Sex by Age_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'}

## Defining a Census data retrieval function

The following function simplifies the process of retrieving data from the Census API. It also enables the user to rename variable fields (e.g. 'B01001_001E') with aliases for those fields (e.g. 'Sex by Age_Estimate!!Total: (B01001_001E)'), but this option is disabled by default. In addition, it allows more than 50 variables to be retrieved at the same time, thus making it easier to retrieve especially large datasets.

[Note: currently, this function only supports data retrieval for the ACS 5-year and 1-year estimates. However, I may add in the ability to retrieve decennial Census data in the future.]

In [11]:
def retrieve_census_data(survey, year, region, key, variable_list,
                         rename_data_fields = False, 
                         field_names_dict = {}):
    '''This function (which I plan to expand) retrieves data from the US
    Census API. It accommodates more than 50 variables.
    
    survey: the survey from which to retrieve data. The only arguments
    currently supported are 'acs5' and 'acs1' (for the American Community 
    Survey 5-Year and 1-Year estimates, respectively).
    
    year: the year for which you wish to retrieve survey data. Note that,
    When region is set to 'acs5', the survey results will include data
    for the 5 years leading up to (and including) the 'year' argument.
    (For example, if you set 'year' to 2022, you'll retrieve ACS5 data
    from 2018 to 2022 (inclusive).)
    
    
    region: The geographic level at which you wish to retrieve data. 
    Examples include 'us', 'state', 'county', 'zip', 'msa' 
    (for metropolitan/micropolitan statistical area data), and 'csa' 
    (for combined statistical area data); 
    however, other regions are supported as well. Consult your survey's 
    API examples page for other options. (For instance, if you wanted to 
    retrieve data by urban area within the 2022 ACS5, you could go to 
    https://api.census.gov/data/2022/acs/acs5/examples.html, then search
    for 'urban area.' The Urban Area URL ends with
    '&for=urban%20area:*&key=YOUR_KEY_GOES_HERE'. Therefore, you'd want to
    use 'urban%20area' as your 'region' argument.)   

    (Note: 'zip' will retrieve results by Zip Code
    Tabulation Area, which are similar to (but not identical to)
    # zip codes. See 
    # https://en.wikipedia.org/wiki/ZIP_Code_Tabulation_Area
    # for more information.
    
    variable_list: The list of variables for which to retrieve data.

    key: your personal Census API key.

    rename_data_fields: set to True to replace column names in your 
    dataset with new entries of your choice.

    field_names_dict: A dictionary that stores the original variable names
    retrieved by the Census (e.g. 'B01001_001E' as keys and your desired
    replacements as values. Example: 
    {'B01001_001E': 'Sex by Age_Estimate!!Total:_B01001_001E',
     'B01001_002E': 'Sex by Age_Estimate!!Total:!!Male:_B01001_002E'}'
     
    '''

    # Using the iteration_utilities library to check for duplicate
    # values within variable_list (which could cause issues later on):
    # The following code is based on
    # https://iteration-utilities.readthedocs.io/en/latest/generated/
    # duplicates.html
    duplicate_variables = list(duplicates(variable_list))
    
    if len(duplicate_variables) > 0:
        raise ValueError(f"The following variables appear more than once \
in your variable list: {duplicate_variables}")
    
    if survey == 'acs5':
        survey_string = 'acs/acs5'

    elif survey == 'acs1':
        survey_string = 'acs/acs1'
    
    else:
        raise ValueError("This survey type is not currently supported by \
                         the function.")

    
    # Converting simplified region names into strings that the API 
    # function will recognize:
    if region == 'zip':
        region = 'zip%20code%20tabulation%20area' # Based on
        # the ZCTA example within
        # https://api.census.gov/data/2022/acs/acs5/examples.html
    
    if region == 'csa':
        region = 'combined%20statistical%20area'
    
    if region == 'msa':
        region = 'metropolitan%20statistical\
%20area/micropolitan%20statistical%20area'

    
    # Only 50 variables can be retrieved from the Census API at a time 
    # using the approach shown in this function. The following code 
    # accommodates this limitation by splitting variable_list into 
    # sublists of up to 49 variables. The data retrieved for the variables 
    # in these sublists will then get merged back together.
    # (49 variables are retrieved at a time instead of 50 because it 
    # appears that the initial 'NAME' variable also counts towards 
    # the 50-variable limit.)
    
    i = 0
       
    while i < len(variable_list): # i.e. while there
        # are still more variables to iterate through
        variable_sublist = variable_list[i:i+49] # This line reads the 
        # next 49 variables from variable_list into a sublist that can 
        # then be\ passed to the API
        # print("variable_sublist:", variable_sublist)
        # Converting the list of variables into a string that can be 
        # passed to the API call:
        # (The Census API guide at
        # https://www.census.gov/content/dam/Census/data/developers/
        # api-user-guide/api-guide.pdf
        # demonstrates how to call multiple census variables at once.)
        variable_string = ','.join(variable_sublist)
        # print("variable_string:",variable_string)
    
        # Retrieving data via the Census API:
        # This line was originally based on an example found in
        # https://api.census.gov/data/2022/acs/acs5/examples.html .
    
        # read_json documentation:
        # https://pandas.pydata.org/pandas-docs/stable/reference/api/
        # pandas.read_json.html

        api_url = f'https://api.census.gov/data/{year}/\
{survey_string}?get=NAME,{variable_string}&for={region}:*&key={key}'
        # print(api_url)
        
        df_results = pd.read_json(api_url)
    
        # At this point, the DataFrame's columns are a list of integers; 
        # the desired column names are stored within the first row. 
        # The following code resolves this issue by setting these row 
        # values as the column values and then deleting this row.
    
        df_results.columns = df_results.iloc[0]
        df_results.drop(0, inplace = True)


        # Determining which merge keys to use when combining API results
        # for different sublists together:
        # This is made more complicated by the fact that results for 
        # different regions will have different identifier
        # columns (e.g. 'NAME', 'county', and 'state' for county data but 
        # only 'NAME' and 'state' for state data). However, we can 
        # accommodate this behavior by simply initializing our list of 
        # merge keys as the set of all columns that are *not* also 
        # variable columns.
        if i == 0: # This step only needs to be performed for our first
            # sublist of variables, since merge keys for other sublists
            # will be identical.
            merge_keys = list(set(df_results.columns) 
              - set(variable_sublist))
            # print("merge_keys:",merge_keys)

        if i == 0: # Since this is the first set 
            # of results, we can initialize df_combined_results 
            # as a copy of df_results.
            df_combined_results = df_results.copy()
        else: # Merging our latest set of results into df_results:
            df_combined_results = df_combined_results.merge(
                df_results, on = merge_keys,
                how = 'outer').copy()
            # Added .copy() here in response to a data fragmentation 
        # warning

        i += 49 
        # Allows the function to iterate through the next 49 variables
        # within variable_list

        
    # Converting variable columns to numeric data types:
    for column in variable_list:
        # print(f"Now converting {column} to a numeric type.")
        df_combined_results[column] = pd.to_numeric(
            df_combined_results[column])
        # pd.to_numeric() allows for either integer or float outputs
        # depending on the nature of the original data.
        # See https://pandas.pydata.org/pandas-docs/stable/reference/api/
        # pandas.to_numeric.html

    # Replacing column names with aliases if requested:
    if rename_data_fields == True:
        df_combined_results.rename(
            columns = field_names_dict, inplace = True)

    # The following for loop moves all of the merge keys (e.g. geographic
    # identifiers) to the left side of the table. This is particularly
    # useful when retrieving longer lists of variables, as otherwise,
    # certain keys can get buried in the middle of the dataset
    for i in range(len(merge_keys)):
        df_combined_results.insert(
            i, merge_keys[i], 
            df_combined_results.pop(merge_keys[i]))

    # Adding a 'Year' column to the left of all existing DataFrame columns:
    # (this will prove particularly
    # helpful when comparing data from different years.)
    df_combined_results.insert(0, 'Year', year)
    
    return df_combined_results

(The following code allowed me to test out retrieve_census_data for a particularly long variable list.)

In [12]:
# test_list = list(df_variables['Name'][0:151])

# test_alias_dict = create_variable_aliases(
#     df_variables = df_variables, 
#     variable_list = test_list)

# test_acs5_data = retrieve_census_data(
#     survey = 'acs5', year = latest_acs5_year, region = 'county',
#     variable_list = test_list, 
#     rename_data_fields = True, 
#     field_names_dict = test_alias_dict, key = key)

# test_acs5_data

Next, we'll define a list of years for which we would like to retrieve Census data. In order to make this code easier to use in future years, I'll define these years as an offset of latest_acs5_year rather than hardcoding them.

In [13]:
years_to_retrieve = [latest_acs5_year - 12, latest_acs5_year -10, 
                     latest_acs5_year - 8,
                     latest_acs5_year - 6,
                     latest_acs5_year - 5,
                     latest_acs5_year]
# American Community Survey 1-year estimates weren't available in 2020,
# so you'll want to remove that year from your list if it happens to be present.
# However, because I decided to use 5-year rather than 1-year estimates,
# I commented out this line.
# if 2020 in years_to_retrieve:
#     years_to_retrieve.remove(2020)
years_to_retrieve

[2010, 2012, 2014, 2016, 2017, 2022]

At this point, it's a good idea to confirm that our three variable codes ('B01001_001E', 'B01001_011E', and 'B01001_035E') had the same meaning for all the years whose data we'll be retrieving. We can do so by running the following code, which retrieves these variables and their corresponding descriptions for all of the years in years_to_retrieve.

(Due to the size of the variables.html page, this code can take a while to run, so I commented it out below.)

In [14]:
# var_meanings_by_year_df_list = []
# for year in years_to_retrieve:
#     df_var_list = pd.read_html(
#         f'https://api.census.gov/data/{year}/acs/acs5/variables.html')[
#     0][['Name', 'Label', 'Concept']].query(
#         "Name in @grad_destinations_variable_list")
#     df_var_list.insert(0, 'Year', year)
#     var_meanings_by_year_df_list.append(df_var_list)
# df_var_meanings_by_year = pd.concat([df for df in var_meanings_by_year_df_list])
# df_var_meanings_by_year.to_csv('var_meanings_by_year.csv', index = False)
# df_var_meanings_by_year

Here's a saved .csv copy of this table that confirms that these codes had the same meaning in each of the years we're analyzing:

In [15]:
pd.read_csv('var_meanings_by_year.csv')

,Year,Name,Label,Concept
0,2010,B01001_001E,Estimate!!Total,SEX BY AGE
1,2010,B01001_011E,Estimate!!Total!!Male!!25 to 29 years,SEX BY AGE
2,2010,B01001_035E,Estimate!!Total!!Female!!25 to 29 years,SEX BY AGE
3,2012,B01001_001E,Estimate!!Total,SEX BY AGE
4,2012,B01001_011E,Estimate!!Total!!Male!!25 to 29 years,SEX BY AGE
5,2012,B01001_035E,Estimate!!Total!!Female!!25 to 29 years,SEX BY AGE
6,2014,B01001_001E,Estimate!!Total,SEX BY AGE
7,2014,B01001_011E,Estimate!!Total!!Male!!25 to 29 years,SEX BY AGE
8,2014,B01001_035E,Estimate!!Total!!Female!!25 to 29 years,SEX BY AGE
9,2016,B01001_001E,Estimate!!Total,SEX BY AGE


We're now ready to retrieve our population totals for the years referenced in years_to_retrieve. We'll do so by calling retrieve_census_data() for each of these years via a for loop, then adding their respective DataFrames together using pd.concat().

(Note: Only 5825 results showed up when I requested ACS1 data (as opposed to over 25,000 results for ACS5 data), so many counties were not getting incorporated within the ACS1 results. Therefore, the ACS5 should be the best survey to use for evaluating county-level growth.)

In [16]:
census_data_by_year_df_list = []
for year in years_to_retrieve:
    df_data = retrieve_census_data(
        survey = 'acs5', year = year, 
        region = 'county',
        variable_list = grad_destinations_variable_list, 
        rename_data_fields = True, field_names_dict = grad_destinations_alias_dict,
        key = key)
    census_data_by_year_df_list.append(df_data)
df_growth_data_by_year = pd.concat(df for df in census_data_by_year_df_list)
# Removing Puerto Rico from our list of results so as to focus only on counties
# and county equivalents within the 50 US states + DC:
df_growth_data_by_year.query("state != '72'", inplace = True)
df_growth_data_by_year

,Year,NAME,state,county,Sex by Age_Estimate!!Total: (B01001_001E),Sex by Age_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E),Sex by Age_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)
16,2010,"Merced County, California",06,047,250699,9255,8600
17,2010,"Modoc County, California",06,049,9605,253,170
18,2010,"Mono County, California",06,051,13905,673,439
19,2010,"Monterey County, California",06,053,407435,17107,14427
20,2010,"Napa County, California",06,055,134051,4286,3816
...,...,...,...,...,...,...,...
3140,2022,"Sweetwater County, Wyoming",56,037,42079,1260,1046
3141,2022,"Teton County, Wyoming",56,039,23346,1104,855
3142,2022,"Uinta County, Wyoming",56,041,20546,621,537
3143,2022,"Washakie County, Wyoming",56,043,7725,181,210


The following cell adds together male and female population totals in order to calculate the total number of 25- to 29-year-olds within each county. It also condenses the original total population column name in order to improve its readability.

In [17]:
df_growth_data_by_year['Total_Pop_25_to_29'] = (df_growth_data_by_year[
'Sex by Age_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)'] + 
df_growth_data_by_year[
'Sex by Age_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'])
df_growth_data_by_year.rename(
    columns = {'Sex by Age_Estimate!!Total: (B01001_001E)':'Total_Pop'}, 
inplace = True)
df_growth_data_by_year.drop(
    ['Sex by Age_Estimate!!Total:!!Male:!!25 to 29 years (B01001_011E)',
     'Sex by Age_Estimate!!Total:!!Female:!!25 to 29 years (B01001_035E)'],
     axis = 1, inplace = True)
                             

df_growth_data_by_year

,Year,NAME,state,county,Total_Pop,Total_Pop_25_to_29
16,2010,"Merced County, California",06,047,250699,17855
17,2010,"Modoc County, California",06,049,9605,423
18,2010,"Mono County, California",06,051,13905,1112
19,2010,"Monterey County, California",06,053,407435,31534
20,2010,"Napa County, California",06,055,134051,8102
...,...,...,...,...,...,...
3140,2022,"Sweetwater County, Wyoming",56,037,42079,2306
3141,2022,"Teton County, Wyoming",56,039,23346,1959
3142,2022,"Uinta County, Wyoming",56,041,20546,1158
3143,2022,"Washakie County, Wyoming",56,043,7725,391


The following code applies pd.pivot() to place all population totals for a given county within the same row, thus making future growth calculations easier.

In [18]:
df_growth_data_by_year_for_pivot = df_growth_data_by_year.copy()
df_growth_data_by_year_pivot = df_growth_data_by_year_for_pivot.pivot(
    columns = 'Year', index = ['NAME', 'county', 'state'],
# The values could be named explicitly, but since pivot() will infer them
    # automatically, there's no need to do so. The following code is thus
    # commented out.
    # values = ['Total_Pop',
#           'Total_Pop_25_to_29']
).reset_index()
df_growth_data_by_year_pivot.columns = df_growth_data_by_year_pivot.columns.to_flat_index()

df_growth_data_by_year_pivot.head()

,"(NAME, )","(county, )","(state, )","(Total_Pop, 2010)","(Total_Pop, 2012)","(Total_Pop, 2014)","(Total_Pop, 2016)","(Total_Pop, 2017)","(Total_Pop, 2022)","(Total_Pop_25_to_29, 2010)","(Total_Pop_25_to_29, 2012)","(Total_Pop_25_to_29, 2014)","(Total_Pop_25_to_29, 2016)","(Total_Pop_25_to_29, 2017)","(Total_Pop_25_to_29, 2022)"
0,"Abbeville County, South Carolina",001,45,25643.0,25387.0,25100.0,24951.0,24788.0,24368.0,1324.0,1198.0,1197.0,1226.0,1290.0,1279.0
1,"Acadia Parish, Louisiana",001,22,61139.0,61611.0,62031.0,62372.0,62607.0,57674.0,4259.0,4188.0,4154.0,4240.0,4163.0,3788.0
2,"Accomack County, Virginia",001,51,34066.0,33454.0,33165.0,33060.0,32840.0,33367.0,1895.0,1846.0,1810.0,1819.0,1602.0,1823.0
3,"Ada County, Idaho",001,16,380718.0,394961.0,409239.0,425798.0,435117.0,497494.0,29381.0,29693.0,29721.0,30646.0,30963.0,34232.0
4,"Adair County, Iowa",001,19,7779.0,7628.0,7543.0,7330.0,7192.0,7479.0,393.0,394.0,376.0,362.0,369.0,397.0


In [19]:
df_growth_data_by_year_pivot.columns = [
    column[0] + '_' + str(column[1]) if isinstance(column[1], int) 
    else column[0] for column in df_growth_data_by_year_pivot.columns]
df_growth_data_by_year_pivot

,NAME,county,state,Total_Pop_2010,Total_Pop_2012,Total_Pop_2014,Total_Pop_2016,Total_Pop_2017,Total_Pop_2022,Total_Pop_25_to_29_2010,Total_Pop_25_to_29_2012,Total_Pop_25_to_29_2014,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2017,Total_Pop_25_to_29_2022
0,"Abbeville County, South Carolina",001,45,25643.0,25387.0,25100.0,24951.0,24788.0,24368.0,1324.0,1198.0,1197.0,1226.0,1290.0,1279.0
1,"Acadia Parish, Louisiana",001,22,61139.0,61611.0,62031.0,62372.0,62607.0,57674.0,4259.0,4188.0,4154.0,4240.0,4163.0,3788.0
2,"Accomack County, Virginia",001,51,34066.0,33454.0,33165.0,33060.0,32840.0,33367.0,1895.0,1846.0,1810.0,1819.0,1602.0,1823.0
3,"Ada County, Idaho",001,16,380718.0,394961.0,409239.0,425798.0,435117.0,497494.0,29381.0,29693.0,29721.0,30646.0,30963.0,34232.0
4,"Adair County, Iowa",001,19,7779.0,7628.0,7543.0,7330.0,7192.0,7479.0,393.0,394.0,376.0,362.0,369.0,397.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3154,"Yuma County, Arizona",027,04,190526.0,196420.0,201453.0,203292.0,204281.0,204374.0,12348.0,13162.0,14074.0,14422.0,14708.0,14683.0
3155,"Yuma County, Colorado",125,08,9896.0,10032.0,10131.0,10150.0,10109.0,9938.0,594.0,588.0,606.0,614.0,617.0,649.0
3156,"Zapata County, Texas",505,48,13609.0,14014.0,14231.0,14335.0,14415.0,13896.0,864.0,919.0,955.0,936.0,930.0,897.0
3157,"Zavala County, Texas",507,48,11658.0,11753.0,12013.0,12107.0,12152.0,9700.0,544.0,750.0,758.0,784.0,713.0,630.0


The following code was derived from my create_enrollment_comparisons() function within  catholic_school_enrollment_trends_v4.ipynb (available at https://github.com/kburchfiel/catholic_school_enrollment/blob/main/catholic_school_enrollment_trends_v4.ipynb).

In [20]:
def create_comparison_fields(df, field_name, year_list,
                             field_year_separator = '_'):
    latest_year = year_list[-1]
    for year in year_list[:-1]:

        # Calculating the nominal change between the two years:
        df[f'{year}-{latest_year} {field_name} Change'] = (
        df[f'{field_name}{field_year_separator}{latest_year}'] 
        - df[f'{field_name}{field_year_separator}{year}'] )

        # Calculating the percentage change:
        df[f'{year}-{latest_year} {field_name} % Change'] = 100*((
        df[f'{field_name}{field_year_separator}{latest_year}'] 
        / df[f'{field_name}{field_year_separator}{year}']) - 1)

    
        # Calculating ranks and percentiles for both the nominal 
        # change and % change columns:
        df[f'{year}-{latest_year} Change Rank'] = df[
        f'{year}-{latest_year} {field_name} Change'].rank(
            ascending=False, method = 'min')
        
        df[f'{year}-{latest_year} Change Percentile'] = 100 * df[
        f'{year}-{latest_year} {field_name} Change'].rank(
            pct=True, ascending=True, method='max')
        
        df[f'{year}-{latest_year} % Change Rank'] = (
            df[f'{year}-{latest_year} {field_name} % Change'].rank(
            ascending=False, method = 'min'))
        
        df[f'{year}-{latest_year} % Change Percentile'] = (
            100 * df[
            f'{year}-{latest_year} {field_name} % Change'].rank(
            pct=True, ascending=True, method='max'))

In [21]:
create_comparison_fields(
    df = df_growth_data_by_year_pivot,
    field_name = 'Total_Pop',
    year_list = years_to_retrieve,
    field_year_separator='_')
                         

In [22]:
df_growth_data_by_year_pivot.head(5)

,NAME,county,state,Total_Pop_2010,Total_Pop_2012,Total_Pop_2014,Total_Pop_2016,Total_Pop_2017,Total_Pop_2022,Total_Pop_25_to_29_2010,Total_Pop_25_to_29_2012,Total_Pop_25_to_29_2014,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2017,Total_Pop_25_to_29_2022,2010-2022 Total_Pop Change,2010-2022 Total_Pop % Change,2010-2022 Change Rank,2010-2022 Change Percentile,2010-2022 % Change Rank,2010-2022 % Change Percentile,2012-2022 Total_Pop Change,2012-2022 Total_Pop % Change,2012-2022 Change Rank,2012-2022 Change Percentile,2012-2022 % Change Rank,2012-2022 % Change Percentile,2014-2022 Total_Pop Change,2014-2022 Total_Pop % Change,2014-2022 Change Rank,2014-2022 Change Percentile,2014-2022 % Change Rank,2014-2022 % Change Percentile,2016-2022 Total_Pop Change,2016-2022 Total_Pop % Change,2016-2022 Change Rank,2016-2022 Change Percentile,2016-2022 % Change Rank,2016-2022 % Change Percentile,2017-2022 Total_Pop Change,2017-2022 Total_Pop % Change,2017-2022 Change Rank,2017-2022 Change Percentile,2017-2022 % Change Rank,2017-2022 % Change Percentile
0,"Abbeville County, South Carolina",001,45,25643.0,25387.0,25100.0,24951.0,24788.0,24368.0,1324.0,1198.0,1197.0,1226.0,1290.0,1279.0,-1275.0,-4.972117,2634.0,15.824808,2294.0,26.694373,-1019.0,-4.013865,2547.0,18.658147,2208.0,29.488818,-732.0,-2.916335,2419.0,22.772277,2087.0,33.375918,-583.0,-2.336580,2412.0,23.045005,2115.0,32.524737,-420.0,-1.694368,2311.0,26.268752,2066.0,34.088733
1,"Acadia Parish, Louisiana",001,22,61139.0,61611.0,62031.0,62372.0,62607.0,57674.0,4259.0,4188.0,4154.0,4240.0,4163.0,3788.0,-3465.0,-5.667414,3000.0,4.124041,2405.0,23.145780,-3937.0,-6.390093,3036.0,3.035144,2574.0,17.795527,-4357.0,-7.023907,3058.0,2.363462,2699.0,13.829447,-4698.0,-7.532226,3093.0,1.308650,2835.0,9.543568,-4933.0,-7.879311,3091.0,1.372486,2899.0,7.500798
2,"Accomack County, Virginia",001,51,34066.0,33454.0,33165.0,33060.0,32840.0,33367.0,1895.0,1846.0,1810.0,1819.0,1602.0,1823.0,-699.0,-2.051899,2323.0,25.767263,1860.0,40.569054,-87.0,-0.260059,1639.0,47.667732,1562.0,50.127796,202.0,0.609076,1318.0,57.936761,1406.0,55.126158,307.0,0.928615,1244.0,60.325567,1356.0,56.750718,527.0,1.604750,1123.0,64.187680,1235.0,60.612831
3,"Ada County, Idaho",001,16,380718.0,394961.0,409239.0,425798.0,435117.0,497494.0,29381.0,29693.0,29721.0,30646.0,30963.0,34232.0,116776.0,30.672571,49.0,98.465473,74.0,97.666240,102533.0,25.960285,48.0,98.498403,64.0,97.987220,88255.0,21.565638,39.0,98.786330,70.0,97.796231,71696.0,16.838031,34.0,98.946696,74.0,97.669965,62377.0,14.335684,30.0,99.074370,79.0,97.510373
4,"Adair County, Iowa",001,19,7779.0,7628.0,7543.0,7330.0,7192.0,7479.0,393.0,394.0,376.0,362.0,369.0,397.0,-300.0,-3.856537,1948.0,37.755754,2138.0,31.681586,-149.0,-1.953330,1723.0,44.984026,1869.0,40.319489,-64.0,-0.848469,1596.0,49.057809,1688.0,46.119451,149.0,2.032742,1373.0,56.208107,1137.0,63.740823,287.0,3.990545,1270.0,59.495691,700.0,77.689116


In [23]:
df_growth_data_by_year_pivot

,NAME,county,state,Total_Pop_2010,Total_Pop_2012,Total_Pop_2014,Total_Pop_2016,Total_Pop_2017,Total_Pop_2022,Total_Pop_25_to_29_2010,Total_Pop_25_to_29_2012,Total_Pop_25_to_29_2014,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2017,Total_Pop_25_to_29_2022,2010-2022 Total_Pop Change,2010-2022 Total_Pop % Change,2010-2022 Change Rank,2010-2022 Change Percentile,2010-2022 % Change Rank,2010-2022 % Change Percentile,2012-2022 Total_Pop Change,2012-2022 Total_Pop % Change,2012-2022 Change Rank,2012-2022 Change Percentile,2012-2022 % Change Rank,2012-2022 % Change Percentile,2014-2022 Total_Pop Change,2014-2022 Total_Pop % Change,2014-2022 Change Rank,2014-2022 Change Percentile,2014-2022 % Change Rank,2014-2022 % Change Percentile,2016-2022 Total_Pop Change,2016-2022 Total_Pop % Change,2016-2022 Change Rank,2016-2022 Change Percentile,2016-2022 % Change Rank,2016-2022 % Change Percentile,2017-2022 Total_Pop Change,2017-2022 Total_Pop % Change,2017-2022 Change Rank,2017-2022 Change Percentile,2017-2022 % Change Rank,2017-2022 % Change Percentile
0,"Abbeville County, South Carolina",001,45,25643.0,25387.0,25100.0,24951.0,24788.0,24368.0,1324.0,1198.0,1197.0,1226.0,1290.0,1279.0,-1275.0,-4.972117,2634.0,15.824808,2294.0,26.694373,-1019.0,-4.013865,2547.0,18.658147,2208.0,29.488818,-732.0,-2.916335,2419.0,22.772277,2087.0,33.375918,-583.0,-2.336580,2412.0,23.045005,2115.0,32.524737,-420.0,-1.694368,2311.0,26.268752,2066.0,34.088733
1,"Acadia Parish, Louisiana",001,22,61139.0,61611.0,62031.0,62372.0,62607.0,57674.0,4259.0,4188.0,4154.0,4240.0,4163.0,3788.0,-3465.0,-5.667414,3000.0,4.124041,2405.0,23.145780,-3937.0,-6.390093,3036.0,3.035144,2574.0,17.795527,-4357.0,-7.023907,3058.0,2.363462,2699.0,13.829447,-4698.0,-7.532226,3093.0,1.308650,2835.0,9.543568,-4933.0,-7.879311,3091.0,1.372486,2899.0,7.500798
2,"Accomack County, Virginia",001,51,34066.0,33454.0,33165.0,33060.0,32840.0,33367.0,1895.0,1846.0,1810.0,1819.0,1602.0,1823.0,-699.0,-2.051899,2323.0,25.767263,1860.0,40.569054,-87.0,-0.260059,1639.0,47.667732,1562.0,50.127796,202.0,0.609076,1318.0,57.936761,1406.0,55.126158,307.0,0.928615,1244.0,60.325567,1356.0,56.750718,527.0,1.604750,1123.0,64.187680,1235.0,60.612831
3,"Ada County, Idaho",001,16,380718.0,394961.0,409239.0,425798.0,435117.0,497494.0,29381.0,29693.0,29721.0,30646.0,30963.0,34232.0,116776.0,30.672571,49.0,98.465473,74.0,97.666240,102533.0,25.960285,48.0,98.498403,64.0,97.987220,88255.0,21.565638,39.0,98.786330,70.0,97.796231,71696.0,16.838031,34.0,98.946696,74.0,97.669965,62377.0,14.335684,30.0,99.074370,79.0,97.510373
4,"Adair County, Iowa",001,19,7779.0,7628.0,7543.0,7330.0,7192.0,7479.0,393.0,394.0,376.0,362.0,369.0,397.0,-300.0,-3.856537,1948.0,37.755754,2138.0,31.681586,-149.0,-1.953330,1723.0,44.984026,1869.0,40.319489,-64.0,-0.848469,1596.0,49.057809,1688.0,46.119451,149.0,2.032742,1373.0,56.208107,1137.0,63.740823,287.0,3.990545,1270.0,59.495691,700.0,77.689116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3154,"Yuma County, Arizona",027,04,190526.0,196420.0,201453.0,203292.0,204281.0,204374.0,12348.0,13162.0,14074.0,14422.0,14708.0,14683.0,13848.0,7.268299,404.0,87.116368,758.0,75.799233,7954.0,4.049486,486.0,84.504792,949.0,69.712460,2921.0,1.449966,666.0,78.760779,1271.0,59.437879,1082.0,0.532239,901.0,71.273540,1445.0,53.909990,93.0,0.045526,1477.0,52.888605,1611.0,48.611554
3155,"Yuma County, Colorado",125,08,9896.0,10032.0,10131.0,10150.0,10109.0,9938.0,594.0,588.0,606.0,614.0,617.0,649.0,42.0,0.424414,1522.0,51.374680,1508.0,51.822251,-94.0,-0.937002,1648.0,47.380192,1678.0,46.421725,-193.0,-1.905044,1820.0,41.903545,1901.0,39.316512,-212.0,-2.088670,1952.0,37.727418,2072.0,33.897223,-171.0,-1.691562,1953.0,37.695500,2064.0,34.152569
3156,"Zapata County, Texas",505,48,13609.0,14014.0,14231.0,14335.0,14415.0,13896.0,864.0,919.0,955.0,936.0,930.0,897.0,287.0,2.108899,1356.

In [24]:
range_for_sorting = 5
sort_column = f'{latest_acs5_year - range_for_sorting}-{latest_acs5_year} \
% Change Rank'

df_growth_data_by_year_pivot.query("Total_Pop_2010 >= 100000").sort_values(
    sort_column).dropna(subset = sort_column)

,NAME,county,state,Total_Pop_2010,Total_Pop_2012,Total_Pop_2014,Total_Pop_2016,Total_Pop_2017,Total_Pop_2022,Total_Pop_25_to_29_2010,Total_Pop_25_to_29_2012,Total_Pop_25_to_29_2014,Total_Pop_25_to_29_2016,Total_Pop_25_to_29_2017,Total_Pop_25_to_29_2022,2010-2022 Total_Pop Change,2010-2022 Total_Pop % Change,2010-2022 Change Rank,2010-2022 Change Percentile,2010-2022 % Change Rank,2010-2022 % Change Percentile,2012-2022 Total_Pop Change,2012-2022 Total_Pop % Change,2012-2022 Change Rank,2012-2022 Change Percentile,2012-2022 % Change Rank,2012-2022 % Change Percentile,2014-2022 Total_Pop Change,2014-2022 Total_Pop % Change,2014-2022 Change Rank,2014-2022 Change Percentile,2014-2022 % Change Rank,2014-2022 % Change Percentile,2016-2022 Total_Pop Change,2016-2022 Total_Pop % Change,2016-2022 Change Rank,2016-2022 Change Percentile,2016-2022 % Change Rank,2016-2022 % Change Percentile,2017-2022 Total_Pop Change,2017-2022 Total_Pop % Change,2017-2022 Change Rank,2017-2022 Change Percentile,2017-2022 % Change Rank,2017-2022 % Change Percentile
619,"Comal County, Texas",091,48,102836.0,108985.0,115808.0,124234.0,129100.0,165201.0,5190.0,5523.0,6028.0,6540.0,6837.0,8964.0,62365.0,60.645105,121.0,96.163683,8.0,99.776215,56216.0,51.581410,115.0,96.357827,4.0,99.904153,49393.0,42.650767,94.0,97.029703,5.0,99.872245,40967.0,32.975675,77.0,97.574210,5.0,99.872327,36101.0,27.963594,69.0,97.829556,10.0,99.712735
1236,"Hays County, Texas",209,48,146439.0,158464.0,170410.0,185686.0,194843.0,245351.0,10049.0,11127.0,12007.0,13332.0,13983.0,16745.0,98912.0,67.544848,58.0,98.177749,6.0,99.840153,86887.0,54.830750,55.0,98.274760,3.0,99.936102,74941.0,43.976879,52.0,98.371127,4.0,99.904184,59665.0,32.132202,43.0,98.659432,6.0,99.840409,50508.0,25.922409,41.0,98.723268,13.0,99.616981
2674,"St. Johns County, Florida",109,12,180624.0,191495.0,203402.0,218362.0,226578.0,278722.0,8510.0,9158.0,9641.0,10689.0,10728.0,11283.0,98098.0,54.310612,60.0,98.113811,15.0,99.552430,87227.0,45.550537,54.0,98.306709,10.0,99.712460,75320.0,37.030118,50.0,98.435005,10.0,99.712552,60360.0,27.642172,42.0,98.691350,16.0,99.521226,52144.0,23.013708,37.0,98.850942,18.0,99.457389
3083,"Williamson County, Texas",491,48,391715.0,426296.0,457218.0,490619.0,508313.0,617396.0,27281.0,29208.0,29869.0,30845.0,31723.0,39167.0,225681.0,57.613571,19.0,99.424552,12.0,99.648338,191100.0,44.828007,16.0,99.520767,12.0,99.648562,160178.0,35.033179,15.0,99.552859,15.0,99.552859,126777.0,25.840214,14.0,99.585062,20.0,99.393553,109083.0,21.459809,13.0,99.616981,25.0,99.233961
2167,"Osceola County, Florida",097,12,258531.0,272355.0,289449.0,311962.0,325168.0,393745.0,17340.0,18138.0,19630.0,21933.0,23383.0,27383.0,135214.0,52.300885,43.0,98.657289,18.0,99.456522,121390.0,44.570505,38.0,98.817891,13.0,99.616613,104296.0,36.032600,33.0,98.977962,13.0,99.616736,81783.0,26.215693,29.0,99.106288,18.0,99.457389,68577.0,21.089714,26.0,99.202043,28.0,99.138206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3014,"Wayne County, North Carolina",191,37,120102.0,122419.0,124093.0,124447.0,124496.0,117480.0,8066.0,8469.0,8720.0,8791.0,8862.0,8297.0,-2622.0,-2.183144,2921.0,6.649616,1886.0,39.737852,-4939.0,-4.034504,3067.0,2.044728,2209.0,29.456869,-6613.0,-5.329068,3098.0,1.085915,2503.0,20.089428,-6967.0,-5.598367,3113.0,0.670284,2654.0,15.320779,-7016.0,-5.635522,3108.0,0.829876,2720.0,13.214172
123,"Baltimore city, Maryland",510,24,620538.0,620644.0,622271.0,621000.0,619796.0,584548.0,55812.0,57765.0,60080.0,61842.0,62299.0,53560.0,-35990.0,-5.799806,3126.0,0.095908,2427.0,22.442455,-36096.0,-5.815894,3129.0,0.063898,2492.0,20.415335,-37723.0,-6.062150,3131.0,0.031939,2608.0,16.735867,-36452.0,-5.869887,3132.0,0.063837,2685.0,14.331312,-35248.0,-5.687033,3132.0,0.063837,2725.0,13.054580
339,"Caddo Parish, Louisiana",017,22,253048.0,254970.0,255529.0,253125.0,251069.0,2

In [25]:
df_1m_pop_pct_changes = df_growth_data_by_year_pivot.query(
    "Total_Pop_2010 >= 1000000").sort_values(sort_column).dropna(
    subset = sort_column).copy()
pd.concat([df_1m_pop_pct_changes.head(), df_1m_pop_pct_changes.tail()])[
['NAME', f'{latest_acs5_year - range_for_sorting}-{latest_acs5_year} \
Total_Pop Change', f'{
latest_acs5_year - range_for_sorting}-{latest_acs5_year} \
Total_Pop % Change',
f'{latest_acs5_year - range_for_sorting}-\
{latest_acs5_year} Change Rank', f'{latest_acs5_year - range_for_sorting}\
-{latest_acs5_year} Change Percentile']]

,NAME,2017-2022 Total_Pop Change,2017-2022 Total_Pop % Change,2017-2022 Change Rank,2017-2022 Change Percentile
2151,"Orange County, Florida",137187.0,10.632871,5.0,99.872327
1275,"Hillsborough County, Florida",117473.0,8.694703,11.0,99.680817
530,"Clark County, Nevada",153490.0,7.266019,4.0,99.904245
2507,"Salt Lake County, Utah",73943.0,6.681395,22.0,99.329716
1792,"Maricopa County, Arizona",275370.0,6.626638,1.0,100.000000
631,"Cook County, Illinois",-13174.0,-0.251482,3124.0,0.319183
2072,"New York County, New York",-8010.0,-0.484317,3115.0,0.606447
1918,"Miami-Dade County, Florida",-14365.0,-0.531525,3126.0,0.255346
277,"Bronx County, New York",-12617.0,-0.866644,3123.0,0.351101
1720,"Los Angeles County, California",-169032.0,-1.672637,3133.0,0.031918


Saving this data as a .csv file so that we can create maps of it within the Mapping section of Python for Nonprofits:

In [26]:
df_growth_data_by_year_pivot.to_csv(
    f'Datasets/grad_destinations_acs_data.csv', 
    index = False)

## Part 2: Retrieving data on marriage and poverty in order to answer Question 2


In [27]:
marriage_poverty_variable_list = [
    'B01001_001E', 'B17010_001E', 'B17010_003E', 'B17010_004E',
    'B17010_011E', 'B17010_016E', 'B17010_017E', 'B17010_023E',
    'B17010_024E', 'B17010_031E', 'B17010_036E', 'B17010_037E',
    'B11003_001E', 'B11003_002E', 'B11003_003E', 'B11004_001E',
    'B11004_002E', 'B11004_003E', 'B17017_002E', 'B17017_004E',
    'B17017_015E', 'B17017_009E', 'B17017_020E', 'B17017_031E',
    'B17017_033E', 'B17017_038E', 'B17017_044E', 'B17017_049E'
]

marriage_poverty_alias_dict = create_variable_aliases(
    df_variables = df_variables, 
    variable_list = marriage_poverty_variable_list)
# marriage_poverty_alias_dict

In [28]:
df_marriage_poverty_acs5_data = retrieve_census_data(
    survey = 'acs5', year = latest_acs5_year, region = 'county',
    variable_list = marriage_poverty_variable_list, 
    rename_data_fields = True, 
    field_names_dict = marriage_poverty_alias_dict, key = key)

# Showing a shortened version of this DataFrame if render_for_pdf
# is set to True so as to prevent its text from getting cut off:

In [29]:
if render_for_pdf == True:
    pd.set_option('display.max_columns', 3)


df_marriage_poverty_acs5_data.head()

,Year,county,state,NAME,Sex by Age_Estimate!!Total: (B01001_001E),Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total: (B17010_001E),Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Married-couple family: (B17010_003E),Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Married-couple family:!!With related children of the householder under 18 years: (B17010_004E),"Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Other family:!!Male householder, no spouse present:!!With related children of the householder under 18 years: (B17010_011E)","Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Other family:!!Female householder, no spouse present: (B17010_016E)","Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Other family:!!Female householder, no spouse present:!!With related children of the householder under 18 years: (B17010_017E)",Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Married-couple family: (B17010_023E),Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Married-couple family:!!With related children of the householder under 18 years: (B17010_024E),"Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Other family:!!Male householder, no spouse present:!!With related children of the householder under 18 years: (B17010_031E)","Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Other family:!!Female householder, no spouse present: (B17010_036E)","Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Other family:!!Female householder, no spouse present:!!With related children of the householder under 18 years: (B17010_037E)",Family Type by Presence and Age of Own Children Under 18 Years_Estimate!!Total: (B11003_001E),Family Type by Presence and Age of Own Children Under 18 Years_Estimate!!Total:!!Married-couple family: (B11003_002E),Family Type by Presence and Age of Own Children Under 18 Years_Estimate!!Total:!!Married-couple family:!!With own children of the householder under 18 years: (B11003_003E),Family Type by Presence and Age of Related Children Under 18 Years_Estimate!!Total: (B11004_001E),Family Type by Presence and Age of Related Children Under 18 Years_Estimate!!Total:!!Married-couple family: (B11004_002E),Family Type by Presence and Age of Related Children Under 18 Years_Estimate!!Total:!!Married-couple family:!!With related children of the householder

In [30]:
# Allowing for larger number of columns to get displayed within
# subsequent DataFrame displays:
if render_for_pdf == True:
    pd.set_option('display.max_columns', 4)

## Performing additional calculations

The following cell uses fields within df_marriage_poverty_acs5_data to calculate poverty rates for:

1. Married-couple households
2. Non-married-couple households
3. Households with 1+ kids below 18 headed by a married couple
4. Households with 1+ kids below 18 *not* headed by a married couple

In addition, it will also calculate differences in poverty rates between:
1. Non-married and married couple households
2. Non-married households with 1+ kids below 18 and married households with 1+ kids below 18

In [31]:
df_marriage_poverty_acs5_data['Non-married-couple households below \
poverty level'] = (df_marriage_poverty_acs5_data[
'Poverty Status in the Past 12 Months by Household Type by \
Age of Householder_Estimate!!Total:!!Income in the past 12 months \
below poverty level: (B17017_002E)'] 
- df_marriage_poverty_acs5_data['Poverty Status in the Past 12 Months \
by Household Type by Age of Householder_Estimate!!Total:!!Income \
in the past 12 months below poverty level:!!Family \
households:!!Married-couple family: (B17017_004E)'])

df_marriage_poverty_acs5_data['Non-married-couple households at or above \
poverty level'] = (df_marriage_poverty_acs5_data[
'Poverty Status in the Past 12 Months by Household Type \
by Age of Householder_Estimate!!Total:!!Income in the past 12 months \
at or above poverty level: (B17017_031E)'
] 
- df_marriage_poverty_acs5_data['Poverty Status in the Past 12 Months \
by Household Type by Age of Householder_Estimate!!Total:!!Income in the \
past 12 months at or above poverty level:!!\
Family households:!!Married-couple family: (B17017_033E)'])

df_marriage_poverty_acs5_data['Non-married households with 1+ kids \
below poverty level'] = (df_marriage_poverty_acs5_data['Poverty Status \
in the Past 12 Months of Families by Family Type by Presence \
of Related Children Under 18 Years by Age of Related Children_Estimate!!\
Total:!!Income in the past 12 months below poverty level:!!Other \
family:!!Male householder, no spouse present:!!With related \
children of the householder under 18 years: (B17010_011E)'] 
+ df_marriage_poverty_acs5_data[
'Poverty Status in the Past 12 Months of Families by Family Type \
by Presence of Related Children Under 18 Years by Age of \
Related Children_Estimate!!Total:!!Income in the past 12 months \
below poverty level:!!Other family:!!Female householder, \
no spouse present:!!With related children \
of the householder under 18 years: (B17010_017E)'])

df_marriage_poverty_acs5_data['Non-married households with 1+ kids \
at or above poverty level'] = (df_marriage_poverty_acs5_data['Poverty \
Status in the Past 12 Months of Families by Family Type by Presence \
of Related Children Under 18 Years by Age of Related Children_Estimate!!\
Total:!!Income in the past 12 months at or above poverty level:!!Other \
family:!!Male householder, no spouse present:!!With related \
children of the householder under 18 years: (B17010_031E)'] 
+ df_marriage_poverty_acs5_data[
'Poverty Status in the Past 12 Months of Families by Family Type \
by Presence of Related Children Under 18 Years by Age of \
Related Children_Estimate!!Total:!!Income in the past 12 months \
at or above poverty level:!!Other family:!!Female householder, \
no spouse present:!!With related children \
of the householder under 18 years: (B17010_037E)'])

df_marriage_poverty_acs5_data[
'% of Married Households Below Poverty Level'] = 100 * (
    df_marriage_poverty_acs5_data['Poverty Status in the Past 12 Months \
by Household Type by Age of Householder_Estimate!!Total:!!Income \
in the past 12 months below poverty level:!!Family \
households:!!Married-couple family: (B17017_004E)'] / 
    (df_marriage_poverty_acs5_data['Poverty Status in the Past 12 Months \
by Household Type by Age of Householder_Estimate!!Total:!!Income in the \
past 12 months at or above poverty level:!!\
Family households:!!Married-couple family: (B17017_033E)'] 
    + df_marriage_poverty_acs5_data['Poverty Status in the Past 12 \
Months by Household Type by Age of Householder_Estimate!!Total:!!Income \
in the past 12 months below poverty level:!!Family \
households:!!Married-couple family: (B17017_004E)']))


df_marriage_poverty_acs5_data[
'% of Non-Married Households Below Poverty Level'] = 100 * (
    df_marriage_poverty_acs5_data['Non-married-couple households below \
poverty level'] / 
    (df_marriage_poverty_acs5_data['Non-married-couple households \
at or above poverty level'] 
+ df_marriage_poverty_acs5_data['Non-married-couple households below \
poverty level']))


df_marriage_poverty_acs5_data['% of Married Households With \
1+ Kids Below Poverty Level'] = 100* (
df_marriage_poverty_acs5_data['Poverty Status in the Past 12 Months \
of Families by Family Type by Presence of Related Children \
Under 18 Years by Age of Related Children_Estimate!!Total:!!Income \
in the past 12 months below poverty level:!!Married-couple \
family:!!With related children of the householder \
under 18 years: (B17010_004E)'] / 
(df_marriage_poverty_acs5_data['Poverty Status in the Past 12 Months \
of Families by Family Type by Presence of Related Children Under \
18 Years by Age of Related Children_Estimate!!Total:!!Income in the \
past 12 months at or above poverty level:!!Married-couple \
family:!!With related children of the householder \
under 18 years: (B17010_024E)'] 
+ df_marriage_poverty_acs5_data['Poverty Status in the Past 12 Months \
of Families by Family Type by Presence of Related Children \
Under 18 Years by Age of Related Children_Estimate!!Total:!!Income \
in the past 12 months below poverty level:!!Married-couple \
family:!!With related children of the householder \
under 18 years: (B17010_004E)']))

df_marriage_poverty_acs5_data[
'% of Non-Married Households With 1+ Kids Below Poverty Level'] = 100 * (
df_marriage_poverty_acs5_data[
'Non-married households with 1+ kids below poverty level'] / (
df_marriage_poverty_acs5_data['Non-married households with 1+ kids \
below poverty level'] +
df_marriage_poverty_acs5_data['Non-married households with 1+ kids \
at or above poverty level']))

# Creating columns that show the difference in poverty rates between
# married and non-married households:

df_marriage_poverty_acs5_data['Non-Married/Married Household \
Poverty Rate Difference'] = (
    df_marriage_poverty_acs5_data[
    '% of Non-Married Households Below Poverty Level'] 
    - df_marriage_poverty_acs5_data[
    '% of Married Households Below Poverty Level'])

df_marriage_poverty_acs5_data['Non-Married Household With 1+ Kids/\
Married Household With 1+ Kids Poverty Rate Difference'] = (
    df_marriage_poverty_acs5_data[
    '% of Non-Married Households With 1+ Kids Below Poverty Level'] 
    - df_marriage_poverty_acs5_data[
    '% of Married Households With 1+ Kids Below Poverty Level'])

# Creating similar columns that show the *ratio* between these two rates:
# (This approach can better adjust for differences in overall poverty 
# rates across counties, but it does have one shortcoming that we'll 
# discuss later in this cell.)

df_marriage_poverty_acs5_data['Non-Married/Married Household \
Poverty Rate Ratio'] = (
    df_marriage_poverty_acs5_data[
    '% of Non-Married Households Below Poverty Level'] 
    / df_marriage_poverty_acs5_data[
    '% of Married Households Below Poverty Level'])

df_marriage_poverty_acs5_data['Non-Married Household With 1+ Kids/\
Married Household With 1+ Kids Poverty Rate Ratio'] = (
    df_marriage_poverty_acs5_data[
    '% of Non-Married Households With 1+ Kids Below Poverty Level'] 
    / df_marriage_poverty_acs5_data['% of Married Households \
With 1+ Kids Below Poverty Level'])

# It appears that dividing by 0 within a DataFrame operation produces
# inf values, which may cause issues during subsequent calculations. 
# Replacing inf values created by the previous operation with NaNs:

df_marriage_poverty_acs5_data.replace(np.inf, np.nan, inplace = True)

df_marriage_poverty_acs5_data.head()

,Year,county,state,NAME,Sex by Age_Estimate!!Total: (B01001_001E),Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total: (B17010_001E),Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Married-couple family: (B17010_003E),Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Married-couple family:!!With related children of the householder under 18 years: (B17010_004E),"Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Other family:!!Male householder, no spouse present:!!With related children of the householder under 18 years: (B17010_011E)","Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Other family:!!Female householder, no spouse present: (B17010_016E)","Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months below poverty level:!!Other family:!!Female householder, no spouse present:!!With related children of the householder under 18 years: (B17010_017E)",Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Married-couple family: (B17010_023E),Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Married-couple family:!!With related children of the householder under 18 years: (B17010_024E),"Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Other family:!!Male householder, no spouse present:!!With related children of the householder under 18 years: (B17010_031E)","Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Other family:!!Female householder, no spouse present: (B17010_036E)","Poverty Status in the Past 12 Months of Families by Family Type by Presence of Related Children Under 18 Years by Age of Related Children_Estimate!!Total:!!Income in the past 12 months at or above poverty level:!!Other family:!!Female householder, no spouse present:!!With related children of the householder under 18 years: (B17010_037E)",Family Type by Presence and Age of Own Children Under 18 Years_Estimate!!Total: (B11003_001E),Family Type by Presence and Age of Own Children Under 18 Years_Estimate!!Total:!!Married-couple family: (B11003_002E),Family Type by Presence and Age of Own Children Under 18 Years_Estimate!!Total:!!Married-couple family:!!With own children of the householder under 18 years: (B11003_003E),Family Type by Presence and Age of Related Children Under 18 Years_Estimate!!Total: (B11004_001E),Family Type by Presence and Age of Related Children Under 18 Years_Estimate!!Total:!!Married-couple family: (B11004_002E),Family Type by Presence and Age of Related Children Under 18 Years_Estimate!!Total:!!Married-couple family:!!With related children of the householder

## Reviewing our marriage and poverty data:

We can use df.describe() to take a quick look at the poverty rate columns that we just calculated. The 50% row is particularly helpful, as it shows the median results in our dataset.

The 2022 copy of these results showed that the median poverty rate by county for married couples was 4.76% compared to 22.04% for non-married couples. Meanwhile, married-couple households with kids had a median poverty rate of 6.17% compared to 31.77% for non-married couple households with kids. Therefore, although we can't determine the *direction* of causation within the marriage/poverty relationship using this data alone, it's evident that married households tend to have lower poverty rates than do non-married households.

We'll create choropleth maps that illustrate some of these data points within the Mapping section of Python for Nonprofits.

In [32]:
df_marriage_poverty_acs5_data[['% of Married Households Below \
Poverty Level',
       '% of Non-Married Households Below Poverty Level',
       '% of Married Households With 1+ Kids Below Poverty Level',
       '% of Non-Married Households With 1+ Kids Below Poverty Level',
       'Non-Married/Married Household Poverty Rate Difference',
       'Non-Married Household With 1+ Kids/Married Household \
With 1+ Kids Poverty Rate Difference',
       'Non-Married/Married Household Poverty Rate Ratio',
       'Non-Married Household With 1+ Kids/Married Household \
With 1+ Kids Poverty Rate Ratio']].describe()

,% of Married Households Below Poverty Level,% of Non-Married Households Below Poverty Level,% of Married Households With 1+ Kids Below Poverty Level,% of Non-Married Households With 1+ Kids Below Poverty Level,Non-Married/Married Household Poverty Rate Difference,Non-Married Household With 1+ Kids/Married Household With 1+ Kids Poverty Rate Difference,Non-Married/Married Household Poverty Rate Ratio,Non-Married Household With 1+ Kids/Married Household With 1+ Kids Poverty Rate Ratio
count,3222.000000,3222.000000,3221.000000,3218.000000,3222.000000,3218.000000,3205.000000,3125.000000
mean,5.985730,23.797786,7.926897,33.429700,17.812056,25.495413,5.394284,6.959111
std,5.168185,9.374700,7.177521,14.799942,6.576608,13.033753,5.982786,9.738970
min,0.000000,0.000000,0.000000,0.000000,-12.225705,-52.000000,0.527273,0.000000
25%,3.203336,17.453112,3.649704,23.834888,13.324396,17.867021,3.436623,3.299141
50%,4.764387,22.041400,6.170981,31.771598,16.947933,24.481687,4.631352,4.805861
75%,6.908005,28.248617,9.789157,41.171320,21.537258,32.149544,6.120610,7.269667
max,52.100457,72.039474,67.663551,100.000000,49.855370,100.000000,202.401575,158.195489


Saving this dataset as a .csv file:

In [33]:
df_marriage_poverty_acs5_data.to_csv(
    f'Datasets/marriage_poverty_acs5_data_{latest_acs5_year}.csv', 
    index = False)

# Appendix

## 1: The Census Python Library

It's worth noting that there is also a 'census' Python library (available via pypi and conda) that helps simplify the process of requesting API data. You may choose to use it for your own Census research, but I ended up not needing it for the data retrieval tasks shown above. In addition, foregoing the library allowed me to demonstrate how to retrieve data directly from an API, which you may find helpful when working with APIs that don't have a corresponding Python library. 

Here's an example of the Census library in use:

In [36]:
## Example of reading data from the Census library into a 
# Pandas DataFrame:
from census import Census
c = Census(key)
pd.DataFrame(c.acs5.get(('NAME', 'B01001_001E'),
{'for': 'county:*'}))

,NAME,B01001_001E,state,county
0,"Autauga County, Alabama",58761.0,01,001
1,"Baldwin County, Alabama",233420.0,01,003
2,"Barbour County, Alabama",24877.0,01,005
3,"Bibb County, Alabama",22251.0,01,007
4,"Blount County, Alabama",59077.0,01,009
...,...,...,...,...
3217,"Vega Baja Municipio, Puerto Rico",54182.0,72,145
3218,"Vieques Municipio, Puerto Rico",8199.0,72,147
3219,"Villalba Municipio, Puerto Rico",21984.0,72,149
3220,"Yabucoa Municipio, Puerto Rico",30313.0,72,151


## 2: The requests library

We can also use Python's *requests* library to retrieve data from the Census API, then convert it to JSON format:

In [37]:
# The following code borrows from the requests library documentation at 
# https://docs.python-requests.org/en/latest/index.html
import requests
r = requests.get(f'https://api.census.gov/data/{latest_acs5_year}/\
acs/acs5?get=NAME,B01001_001E&for=county:*&key={key}')
# Printing the first 300 characters of this output:
print("r.text:\n",r.text[0:300],'\n')
# Printing the first 5 lines of r.json:
print("r.json:\n",r.json()[0:5],'\n')

r.text:
 [["NAME","B01001_001E","state","county"],
["Autauga County, Alabama","58761","01","001"],
["Baldwin County, Alabama","233420","01","003"],
["Barbour County, Alabama","24877","01","005"],
["Bibb County, Alabama","22251","01","007"],
["Blount County, Alabama","59077","01","009"],
["Bullock County, Ala 

r.json:
 [['NAME', 'B01001_001E', 'state', 'county'], ['Autauga County, Alabama', '58761', '01', '001'], ['Baldwin County, Alabama', '233420', '01', '003'], ['Barbour County, Alabama', '24877', '01', '005'], ['Bibb County, Alabama', '22251', '01', '007']] 



Converting our response to JSON allows it to be easily read into a Pandas DataFrame, as shown below:

In [38]:
pd.DataFrame(r.json()).head()
# Note that pd.DataFrame(r.text) would produce the following error:
# "ValueError: DataFrame constructor not properly called!"

,0,1,2,3
0,NAME,B01001_001E,state,county
1,"Autauga County, Alabama",58761,01,001
2,"Baldwin County, Alabama",233420,01,003
3,"Barbour County, Alabama",24877,01,005
4,"Bibb County, Alabama",22251,01,007


I included this approach in the appendix because you may find the requests library useful for other online data retrieval tasks. However, our use of `pd.read_json()` to import Census data rendered an explicit call to the requests library unnecessary.

In [39]:
program_end_time = time.time()
run_time = round(program_end_time - program_start_time, 3)
print(f"Finished running script in {run_time} seconds.")

Finished running script in 97.159 seconds.
